# Entrenar YOLO para contar tornillos y rondanas (Google Colab)

Notebook listo para entrenar tu modelo con **GPU gratis** de Colab.

## Antes de empezar
1. **Runtime → Change runtime type → Hardware accelerator: GPU** (T4) → Save.
2. Ten tu dataset etiquetado y exportado en Roboflow como **YOLOv11**.
3. Ejecuta las celdas en orden con **Shift+Enter**.

Al final descargas `best.pt` y lo usas en tu PC con:
`python conteo_banda.py --modelo best.pt --fuente 0 --clases tornillo,rondana`

## 1. Comprobar que hay GPU

In [ ]:
!nvidia-smi

Si dice *command not found* o no muestra una GPU, vuelve a **Runtime → Change runtime type → GPU**.

## 2. Instalar dependencias

In [ ]:
!pip install -q ultralytics roboflow
import ultralytics; ultralytics.checks()

## 3. Traer el dataset

Usa **una** de las dos opciones (A o B).

### Opción A — descargar con API key  (ALTERNATIVA, opcional)

Solo si **no** quieres subir el zip a mano. Necesitas tu Private API Key de Roboflow.
Como tú ya descargaste el zip, **salta esta celda y usa la Opción B de abajo.**

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="PEGA_TU_API_KEY")            # <-- tu Private API Key de Roboflow
project = rf.workspace("fastener-urest").project("bolts-v5qf3")
dataset = project.version(1).download("yolov11")    # <-- ajusta el numero de version (mira la pagina)

DATA_DIR = dataset.location
print("Dataset descargado en:", DATA_DIR)

# Clases REALES del dataset: usa estos nombres luego en --clases al contar.
!cat {DATA_DIR}/data.yaml

### Opción B — subir el ZIP que descargaste  ✅ (TU CASO)

Ya bajaste **`BOLTS.v1i.yolov11.zip`**, así que usa **esta** opción e ignora la Opción A de arriba.

Al ejecutar la celda aparece un botón **"Elegir archivos"**: selecciona tu zip.
La celda lo sube, lo descomprime y **corrige solo las rutas** del `data.yaml`.

Clases del dataset: `bolt`, `lockwasher`, `nut`, `other`, `screw`, `washer`
(tornillos = `bolt`/`screw`, rondanas = `washer`/`lockwasher`).

In [ ]:
# Sube y prepara el dataset que descargaste de Roboflow.
from google.colab import files
import zipfile, os, glob, yaml

print("Elige tu archivo BOLTS.v1i.yolov11.zip ...")
subido = files.upload()                 # boton para seleccionar el zip
zipname = next(iter(subido))            # nombre del zip subido

DATA_DIR = "/content/dataset"
with zipfile.ZipFile(zipname) as z:
    z.extractall(DATA_DIR)

# Roboflow usa rutas relativas con '../' que fallan en Colab: las dejamos absolutas.
yaml_path = os.path.join(DATA_DIR, "data.yaml")
d = yaml.safe_load(open(yaml_path))
d["path"] = DATA_DIR
d["train"] = "train/images"
d["val"] = "valid/images"
if os.path.isdir(os.path.join(DATA_DIR, "test", "images")):
    d["test"] = "test/images"
yaml.safe_dump(d, open(yaml_path, "w"), sort_keys=False)

print("Clases:", d["names"])
print("Imagenes de entrenamiento:", len(glob.glob(f"{DATA_DIR}/train/images/*")))
print("DATA_DIR =", DATA_DIR)

## 4. Entrenar

Parte de `yolo11n.pt` (rápido). Para más precisión usa `yolo11s.pt`.
Si las piezas se ven muy pequeñas, sube `imgsz` a 960.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")
model.train(
    data=f"{DATA_DIR}/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    name="tornillos_rondanas",
    patience=30,
    plots=True,
)

## 5. Ver resultados del entrenamiento

Curvas de pérdida/precisión y matriz de confusión.

In [ ]:
from IPython.display import Image, display

base = "runs/detect/tornillos_rondanas"
for nombre in ["results.png", "confusion_matrix.png"]:
    ruta = f"{base}/{nombre}"
    print(ruta)
    display(Image(filename=ruta))

## 6. Probar el modelo en imágenes de validación

In [ ]:
import glob
from IPython.display import Image, display

mejor = YOLO("runs/detect/tornillos_rondanas/weights/best.pt")
imagenes = glob.glob(f"{DATA_DIR}/valid/images/*.jpg")[:3]
resultados = mejor.predict(imagenes, conf=0.25, save=True)

for ruta in glob.glob(f"{resultados[0].save_dir}/*.jpg")[:3]:
    display(Image(filename=ruta))

## 7. Descargar el modelo entrenado

Guarda `best.pt` en tu PC y cópialo a la carpeta del proyecto `computo`.

In [ ]:
from google.colab import files
files.download("runs/detect/tornillos_rondanas/weights/best.pt")

## 8. Usarlo en tu PC

Copia `best.pt` a la carpeta `computo` y ejecuta (usa los nombres de clase reales del dataset):

```bash
# Contar tornillos y rondanas en vivo
python conteo_banda.py --modelo best.pt --fuente 0 --clases bolt,screw,washer,lockwasher

# Sobre un video, guardando el resultado
python conteo_banda.py --modelo best.pt --fuente piezas.mp4 --clases bolt,screw,washer,lockwasher --guardar salida.mp4

# Solo rondanas
python conteo_banda.py --modelo best.pt --fuente piezas.mp4 --clases washer,lockwasher
```

Clases disponibles: `bolt`, `lockwasher`, `nut`, `other`, `screw`, `washer`.
El resto del sistema (seguimiento + conteo por línea) no cambia.